# Modeling & Explainability

Ce notebook relit le benchmark, le seuil opérationnel, la calibration et les artefacts SHAP. La priorité métier reste la détection des churners, mais la précision et la calibration encadrent le risque de sur-sollicitation.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

root = Path('..').resolve()
metrics = root / 'outputs' / 'metrics'
figures = root / 'reports' / 'figures'
summary = json.loads((metrics / 'model_selection_summary.json').read_text())
stability = json.loads((metrics / 'model_stability.json').read_text())
benchmark = pd.read_csv(metrics / 'model_benchmark.csv')
summary, stability['gaps_test_minus_validation']

In [ ]:
benchmark[['candidate_name', 'model_family', 'imbalance_strategy', 'validation_pr_auc', 'validation_recall', 'validation_precision']]

## Threshold and Calibration

Le seuil n'est pas fixé à 0,50 par défaut. Il est choisi pour soutenir une stratégie de rétention où manquer un churner est plus coûteux qu'une fausse alerte raisonnable. La calibration reste analysée séparément car un bon classement n'implique pas toujours une probabilité parfaitement calibrée.

In [ ]:
threshold_sensitivity = pd.read_csv(metrics / 'threshold_sensitivity.csv')
calibration = pd.read_csv(metrics / 'calibration_table.csv')
threshold_sensitivity.iloc[(threshold_sensitivity['threshold'] - summary['threshold_policy']['threshold']).abs().idxmin()], calibration

In [ ]:
for figure_name in ['validation_model_curves.png', 'test_model_curves.png', 'shap_summary.png', 'shap_bar.png']:
    display(Image(filename=figures / figure_name))

## Modeling Takeaways

- XGBoost pondéré reste le meilleur compromis V1/V2 pour ce dataset.
- Les métriques très élevées doivent être accompagnées de garde-fous : stabilité validation/test, calibration et slices.
- SHAP aide à expliquer le score, mais ne doit pas être lu comme une preuve causale.